# Student Task: Build a RAG Question-Answering System

## Goal
In this task, you will build a Retrieval-Augmented Generation (RAG) pipeline over the provided `RAG and Fine-tuning - Final.pdf` file.

You will implement these stages:

```text
PDF -> Text -> Chunks -> Embeddings -> Retrieval -> Augmented Prompt -> Grounded Answer
```

Complete every section marked `TODO`.

## Learning objectives

By the end of this task, you should be able to:

- Explain why RAG is useful for private, changing, or large information sources.
- Load and prepare a PDF for semantic search.
- Split a document into overlapping chunks.
- Create embeddings and retrieve the most relevant chunks with cosine similarity.
- Augment a prompt with retrieved context.
- Generate an answer that is grounded in the source document.
- Compare RAG with fine-tuning and describe the purpose of LoRA/QLoRA.

## Task requirements

Your final notebook should:

1. Read the supplied PDF with `pypdf`.
2. Create chunks with a configurable size and overlap.
3. Embed all chunks and the user's question with OpenAI.
4. Retrieve the top 3 relevant chunks using cosine similarity.
5. Generate a concise answer using only the retrieved context.
6. Print the retrieved sources, answer, and a simple grounding check.
7. Answer the conceptual questions near the end.

Do not place your API key directly in the notebook.

## 1. Setup

In [1]:
# !pip install -q -U openai pypdf numpy python-dotenv

import os
from getpass import getpass
from pathlib import Path
import re
import numpy as np
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pypdf import PdfReader

dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path, override=True)
api_key = os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
CHAT_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = Path("F10_GP2_REPORT.pdf")

## 2. Load the source document

In [2]:
# 1: Read every page from PDF_PATH and combine the extracted text.
# Save the result in document_text and print its character count.
reader = PdfReader(PDF_PATH)
document_text = "\n".join(page.extract_text() or "" for page in reader.pages)

print("Document characters:", len(document_text) if document_text else 0)

Document characters: 293794


## 3. Chunk the document

Chunking makes retrieval possible because the system can search meaningful sections instead of sending the entire PDF to the model. Keep the overlap so ideas split across boundaries are less likely to be lost.

In [3]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# 2: Write chunk_text(text, chunk_size, overlap).
# Return a list of non-empty overlapping text chunks.
def chunk_text(text, chunk_size=900, overlap=150):
    step = max(chunk_size - overlap, 1)
    pieces = []
    for start in range(0, len(text), step):
        piece = text[start:start + chunk_size].strip()
        if piece:
            pieces.append(piece)
    return pieces

chunks = chunk_text(document_text, CHUNK_SIZE, CHUNK_OVERLAP)
print("Number of chunks:", len(chunks))
print("First chunk preview:\n", chunks[0][:500])

Number of chunks: 392
First chunk preview:
 IV 
 
Acknowledgement 
We are truly grateful to everyone who supported us during this project. 
Our deepest thanks go to Dr. Lamees Alhazzaa, our supervisor at Imam Mohammad Ibn Saud 
Islamic University. Her guidance and constant encouragement pushed us to do our best work. We 
also extend our gratitude and are proud to be a part of the Computer Science Department  at 
Imam Mohammad Ibn Saud Islamic University , which provided the ideal environment for us 
to learn and build. 
We extend a specia


## 4. Create embeddings and an in-memory vector index

The embedding represents meaning as a vector. The index below is intentionally simple: it stores vectors in NumPy and uses cosine similarity.

In [4]:
# 3: Complete embed_texts so it returns one list of embedding vectors per input text.
def embed_texts(texts):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in sorted(response.data, key=lambda item: item.index)]

# 4: Complete cosine_similarity_matrix.
# It should return the cosine similarity between one query vector and every row in matrix.
def cosine_similarity_matrix(query_vector, matrix):
    query_vector = np.asarray(query_vector, dtype=np.float32)
    matrix = np.asarray(matrix, dtype=np.float32)
    query_norm = np.linalg.norm(query_vector)
    matrix_norms = np.linalg.norm(matrix, axis=1)
    return (matrix @ query_vector) / np.maximum(matrix_norms * query_norm, 1e-10)

chunk_embeddings = np.array(embed_texts(chunks), dtype=np.float32)
print("Embedding matrix shape:", chunk_embeddings.shape)

Embedding matrix shape: (392, 1536)


## 5. Retrieve relevant context

In [5]:
question = "What is the difference between RAG and fine-tuning, and when should each be used?"
TOP_K = 3

# 5: Embed the question, calculate similarities, and select the top TOP_K chunks.
# Save the selected text in retrieved_chunks and their scores in retrieved_scores.
query_embedding = np.array(embed_texts([question])[0], dtype=np.float32)
similarities = cosine_similarity_matrix(query_embedding, chunk_embeddings)
top_indices = np.argsort(similarities)[::-1][:TOP_K]
retrieved_chunks = [chunks[i] for i in top_indices]
retrieved_scores = [float(similarities[i]) for i in top_indices]

for rank, (score, chunk) in enumerate(zip(retrieved_scores, retrieved_chunks), start=1):
    print(f"[{rank}] score={score:.3f}\n{chunk[:400]}\n")

[1] score=0.454
for low -
resource and accent-diverse scenarios. Similarly, Yang et al. in [54] proposed Parameter-Efficient 
Learning for Mandarin accent adaptation. Their method combines model reprogramming which 
subtly perturbs inputs with optimal transport regularization and residual adapters, updating only 
0.6–1.2% of model parameters. It achieves naturalness and accent quality equal to full fine-tuning, 


[2] score=0.385
n 6.6  provides a conclusion 
summarizing the key evaluation findings and identified improvement areas. 
 
6.2 Model Performance Evaluation 
The performance of the fine -tuned XTTS model was evaluated using a combination of 
objective training metrics and subjective listening assessments. Three emotions were evaluated 
lively, casual, and serious to examine consistency in speech quality, dialect a

[3] score=0.374
ro-shot model using filler tokens and flow-
matching for spectrogram generation, offering an efficient architecture without sacrificing output 
qual

In [6]:
question = "Why is Arabic difficult for text-to-speech systems?"
TOP_K = 3

query_embedding = np.array(embed_texts([question])[0], dtype=np.float32)
similarities = cosine_similarity_matrix(query_embedding, chunk_embeddings)
top_indices = np.argsort(similarities)[::-1][:TOP_K]
retrieved_chunks = [chunks[i] for i in top_indices]
retrieved_scores = [float(similarities[i]) for i in top_indices]

for rank, (score, chunk) in enumerate(zip(retrieved_scores, retrieved_chunks), start=1):
    print(f"[{rank}] score={score:.3f}\n{chunk[:400]}\n")

[1] score=0.723
2.3.3.3 Arabic Systems 
Early surveys and papers point out that Arabic TTS has lagged behind English and other 
languages. Al Masri et al. in [107] note as early as April 2022 that very few high -quality Arabic 
TTS systems exist, and most improvements target other languages. A March 2023 survey of 36 
studies from 2000–2022  [25] also shows that most Arabic systems still use traditional methods, 

[2] score=0.717
eavily influences the pronunciation of 
standard Arabic. Another challenge mentione d is the infrequent use of diacritical marks in non -
academic writing, which are crucial for differentiating word sounds and meanings in Arabic. These 
complexities introduced by different dialects and the nature of written Arabic create significant 
hurdles in developing high-quality Arabic TTS systems. 
 
2.2.3.

[3] score=0.670
y.  
To improve inference speed and stability,  non-autoregressive models have been introduced. 
These include FastSpeech2 [23], which removes autor

## 6. Augment the prompt and generate a grounded answer

The model must use the retrieved context and must say when the answer is not supported by the source.

In [7]:
# 6: Build context from retrieved_chunks.
# 7: Write an augmented prompt containing the question and context.
context = "\n\n---\n\n".join(retrieved_chunks)
augmented_prompt = (
    "Answer the question using only the context below.\n"
    "If the context does not contain the answer, say that the source does not cover it.\n\n"
    f"Question: {question}\n\n"
    f"Context:\n{context}"
)

# 8: Call the Responses API and save the answer in answer.
response = client.responses.create(model=CHAT_MODEL, input=augmented_prompt)
answer = response.output_text

print("Answer:")
print(answer)

Answer:
Arabic is difficult for text-to-speech (TTS) systems due to several reasons highlighted in the context:

1. **Dialectal Variation:** Arabic consists of diverse dialects (e.g., North African, Gulf, Levantine) which differ significantly, impacting pronunciation. Most systems focus on a few dialects, leading to challenges in broader coverage.

2. **Missing Diacritics:** The infrequent use of diacritical marks in written Arabic, especially outside academic contexts, makes it difficult to correctly determine word sounds and meanings, as these marks differentiate pronunciation.

3. **Complex Morphology:** Arabic’s complex morphological structure adds difficulty for TTS systems to generate accurate speech.

4. **Limited High-Quality Resources:** There is a scarcity of open speech datasets and shared dialect resources, along with inconsistent evaluation methods.

5. **Lagging System Development:** Arabic TTS development trails behind other languages, with many systems still using tradi

## 7. Evaluate grounding

This is a small educational check, not a complete evaluation system. Inspect whether important terms from the retrieved context appear in the answer and whether the answer admits missing evidence.

In [8]:
# 9: Normalize text and calculate how many distinctive context terms appear in the answer.
COMMON_WORDS = {
    "that", "this", "these", "those", "with", "from", "have", "having", "been",
    "were", "they", "their", "there", "which", "when", "what", "where", "will",
    "would", "could", "should", "then", "than", "also", "into", "such", "about",
    "each", "other", "some", "only", "more", "most", "both", "does", "used",
    "using", "between", "because", "while", "through", "however", "therefore",
}

def normalize_words(text):
    words = set(re.findall(r"[a-zA-Z]{4,}", text.lower()))
    return words - COMMON_WORDS

context_terms = normalize_words(context or "")
answer_terms = normalize_words(answer or "")
overlap = context_terms & answer_terms
grounding_ratio = len(overlap) / max(len(answer_terms), 1)

print(f"Distinctive-term overlap: {len(overlap)}")
print(f"Simple grounding ratio: {grounding_ratio:.2%}")
print("Retrieved context used:", bool(context))

Distinctive-term overlap: 48
Simple grounding ratio: 56.47%
Retrieved context used: True


## 8. Conceptual reflection

1. What do indexing, chunking, retrieval, and augmentation each do in a RAG workflow?
   - chunking: cuts the document into small passages
   - indexing: stores each passage as a vector so it can be searched by meaning
   - retrieval: picks the passages closest to the question
   - augmentation: pastes those passages into the prompt before the model answers
2. Why can poor retrieval produce a poor answer even when the language model is capable?
   - the model only sees what retrieval hands it
   - wrong chunks mean the answer is never in the prompt
   - it then guesses or refuses, no matter how good it is
3. How is a vector database different from a traditional keyword database?
   - keyword matches exact words, vector matches meaning
   - so paraphrases and synonyms still hit
   - it returns nearest neighbours by distance, not exact rows
4. When is RAG a better choice than fine-tuning? When might fine-tuning be better?
   - rag: facts that change, private documents, answers that must cite a source
   - fine-tuning: fixed style, format, or behaviour you want baked in
   - fine-tuning teaches how to answer, not what is currently true
5. What do LoRA and QLoRA change during fine-tuning, and why are base model weights often frozen?
   - lora: trains small adapter matrices instead of the whole model
   - qlora: same, with the base model quantized to 4-bit to fit smaller gpus
   - freezing keeps the general knowledge, cuts cost, and lets you swap adapters per task


## Submission checklist

- [x] All `TODO` sections are completed.
- [x] The notebook runs from top to bottom without errors.
- [x] The PDF is loaded and split into overlapping chunks.
- [x] The top retrieved chunks and similarity scores are printed.
- [x] The answer is generated from retrieved context.
- [ ] The grounding check is printed and interpreted.
- [x] The conceptual reflection questions are answered.
- [x] No API key is written directly in the notebook.